In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy rustworkx scipy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Algoritma pengoptimuman hampiran kuantum

*Anggaran penggunaan: 22 minit pada pemproses Heron r3 (NOTA: Ini hanyalah anggaran. Masa jalan anda mungkin berbeza.)*
## Latar Belakang
Tutorial ini menunjukkan cara melaksanakan **Algoritma Pengoptimuman Hampiran Kuantum (QAOA)** – kaedah iteratif hibrid (kuantum-klasik) – dalam konteks corak Qiskit. Anda akan menyelesaikan masalah **Pemotongan Maksimum** (atau **Max-Cut**) untuk graf kecil terlebih dahulu, kemudian belajar cara melaksanakannya pada skala utiliti. Semua pelaksanaan perkakasan dalam tutorial ini sepatutnya dapat dijalankan dalam had masa Pelan Terbuka yang boleh diakses secara percuma.

Masalah Max-Cut ialah masalah pengoptimuman yang sukar diselesaikan (lebih khusus, ia adalah masalah *NP-hard*) dengan pelbagai aplikasi dalam pengelompokan, sains rangkaian, dan fizik statistik. Tutorial ini mempertimbangkan graf nod yang disambungkan oleh tepi, dan bertujuan untuk membahagikan nod kepada dua set supaya bilangan tepi yang dilalui potongan ini dimaksimumkan.

![Ilustrasi masalah max-cut](../docs/images/tutorials/quantum-approximate-optimization-algorithm/maxcut-illustration.avif)
## Keperluan
Sebelum memulakan tutorial ini, pastikan anda telah memasang perkara berikut:
- Qiskit SDK v1.0 atau lebih baharu, dengan sokongan [visualisasi](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.22 atau lebih baharu (`pip install qiskit-ibm-runtime`)

Selain itu, anda perlu mengakses instans pada [IBM Quantum Platform](/guides/cloud-setup). Perhatikan bahawa tutorial ini tidak boleh dilaksanakan pada Pelan Terbuka, kerana ia menjalankan beban kerja menggunakan [sesi](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/session), yang hanya tersedia dengan akses Pelan Premium.
## Persediaan

In [1]:
import matplotlib
import matplotlib.pyplot as plt
import rustworkx as rx
from rustworkx.visualization import mpl_draw as draw_graph
import numpy as np
from scipy.optimize import minimize
from collections import defaultdict
from typing import Sequence


from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import QAOAAnsatz
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import Session, EstimatorV2 as Estimator
from qiskit_ibm_runtime import SamplerV2 as Sampler

## Bahagian I. QAOA skala kecil
Bahagian pertama tutorial ini menggunakan masalah Max-Cut berskala kecil untuk menggambarkan langkah-langkah menyelesaikan masalah pengoptimuman menggunakan komputer kuantum.

Untuk memberikan konteks sebelum memetakan masalah ini kepada algoritma kuantum, anda boleh lebih memahami bagaimana masalah Max-Cut menjadi masalah pengoptimuman kombinatorial klasik dengan terlebih dahulu mempertimbangkan peminimuman fungsi $f(x)$

$$
\min_{x\in {0, 1}^n}f(x),
$$

di mana input $x$ ialah vektor yang komponennya sepadan dengan setiap nod dalam graf. Kemudian, hadkan setiap komponen ini kepada sama ada $0$ atau $1$ (yang mewakili sama ada ia dimasukkan atau tidak dalam potongan). Kes contoh berskala kecil ini menggunakan graf dengan $n=5$ nod.

Anda boleh menulis fungsi pasangan nod $i,j$ yang menunjukkan sama ada tepi $(i,j)$ yang sepadan berada dalam potongan. Sebagai contoh, fungsi $x_i + x_j - 2 x_i x_j$ bernilai 1 hanya jika salah satu daripada $x_i$ atau $x_j$ adalah 1 (yang bermaksud tepi berada dalam potongan) dan sifar sebaliknya. Masalah memaksimumkan tepi dalam potongan boleh diformulasikan sebagai

$$
\max_{x\in {0, 1}^n} \sum_{(i,j)} x_i + x_j - 2 x_i x_j,
$$

yang boleh ditulis semula sebagai peminimuman dalam bentuk

$$
\min_{x\in {0, 1}^n} \sum_{(i,j)}  2 x_i x_j - x_i - x_j.
$$

Minimum $f(x)$ dalam kes ini adalah apabila bilangan tepi yang dilalui potongan adalah maksimum. Seperti yang anda lihat, tiada apa-apa yang berkaitan dengan pengkomputeran kuantum lagi. Anda perlu memformulasikan semula masalah ini kepada sesuatu yang boleh difahami oleh komputer kuantum.
Mulakan masalah anda dengan mencipta graf dengan $n=5$ nod.

In [2]:
n = 5

graph = rx.PyGraph()
graph.add_nodes_from(np.arange(0, n, 1))
edge_list = [
    (0, 1, 1.0),
    (0, 2, 1.0),
    (0, 4, 1.0),
    (1, 2, 1.0),
    (2, 3, 1.0),
    (3, 4, 1.0),
]
graph.add_edges_from(edge_list)
draw_graph(graph, node_size=600, with_labels=True)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/6ced6bea-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/6ced6bea-0.avif)

### Langkah 1: Petakan input klasik kepada masalah kuantum
Langkah pertama corak ini ialah memetakan masalah klasik (graf) kepada **litar** dan **operator** kuantum. Untuk melakukan ini, terdapat tiga langkah utama yang perlu diambil:

1. Gunakan siri reformulasi matematik untuk mewakili masalah ini menggunakan notasi masalah Pengoptimuman Binari Tak Berkekangan Kuadratik (QUBO).
2. Tulis semula masalah pengoptimuman sebagai Hamiltonian yang keadaan asasnya sepadan dengan penyelesaian yang meminimumkan fungsi kos.
3. Cipta litar kuantum yang akan menyediakan keadaan asas Hamiltonian ini melalui proses yang serupa dengan penganilan kuantum.

**Nota:** Dalam metodologi QAOA, anda akhirnya ingin mempunyai operator (**Hamiltonian**) yang mewakili **fungsi kos** algoritma hibrid kita, serta litar terparameter (**Ansatz**) yang mewakili keadaan kuantum dengan penyelesaian calon kepada masalah tersebut. Anda boleh mengambil sampel daripada keadaan calon ini dan kemudian menilainya menggunakan fungsi kos.

#### Graf &rarr; masalah pengoptimuman
Langkah pertama pemetaan ialah perubahan notasi. Berikut menyatakan masalah dalam notasi QUBO:

$$
\min_{x\in {0, 1}^n}x^T Q x,
$$

di mana $Q$ ialah matriks $n\times n$ nombor nyata, $n$ sepadan dengan bilangan nod dalam graf anda, $x$ ialah vektor pemboleh ubah binari yang diperkenalkan di atas, dan $x^T$ menunjukkan transpos vektor $x$.

```
Maximize
 -2*x_0*x_1 - 2*x_0*x_2 - 2*x_0*x_4 - 2*x_1*x_2 - 2*x_2*x_3 - 2*x_3*x_4 + 3*x_0
 + 2*x_1 + 3*x_2 + 2*x_3 + 2*x_4

Subject to
  No constraints

  Binary variables (5)
    x_0 x_1 x_2 x_3 x_4
```
### Masalah pengoptimuman &rarr; Hamiltonian
Anda kemudian boleh memformulasikan semula masalah QUBO sebagai **Hamiltonian** (di sini, matriks yang mewakili tenaga suatu sistem):

$$
H_C=\sum_{ij}Q_{ij}Z_iZ_j + \sum_i b_iZ_i.
$$

<details>
<summary>
**Langkah reformulasi daripada masalah QAOA kepada Hamiltonian**
</summary>

Untuk menunjukkan bagaimana masalah QAOA boleh ditulis semula dengan cara ini, pertama gantikan pemboleh ubah binari $x_i$ dengan set pemboleh ubah baru $z_i\in{-1, 1}$ melalui

$$
x_i = \frac{1-z_i}{2}.
$$

Di sini anda dapat lihat bahawa jika $x_i$ adalah $0$, maka $z_i$ mesti $1$. Apabila $x_i$ digantikan dengan $z_i$ dalam masalah pengoptimuman ($x^TQx$), formulasi yang setara boleh diperoleh.

$$
x^TQx=\sum_{ij}Q_{ij}x_ix_j \\ =\frac{1}{4}\sum_{ij}Q_{ij}(1-z_i)(1-z_j) \\=\frac{1}{4}\sum_{ij}Q_{ij}z_iz_j-\frac{1}{4}\sum_{ij}(Q_{ij}+Q_{ji})z_i + \frac{n^2}{4}.
$$

Sekarang jika kita takrifkan $b_i=-\sum_{j}(Q_{ij}+Q_{ji})$, buang faktor pra, dan sebutan pemalar $n^2$, kita tiba pada dua formulasi setara bagi masalah pengoptimuman yang sama.

$$
\min_{x\in{0,1}^n} x^TQx\Longleftrightarrow \min_{z\in{-1,1}^n}z^TQz + b^Tz
$$

Di sini, $b$ bergantung kepada $Q$. Perhatikan bahawa untuk mendapatkan $z^TQz + b^Tz$ kita menggugurkan faktor 1/4 dan ofset pemalar $n^2$ yang tidak memainkan peranan dalam pengoptimuman.

Sekarang, untuk mendapatkan formulasi kuantum masalah tersebut, naikkan pemboleh ubah $z_i$ kepada matriks Pauli $Z$, seperti matriks $2\times 2$ dalam bentuk

$$
Z_i = \begin{pmatrix}1 & 0 \\ 0 & -1\end{pmatrix}.
$$

Apabila anda menggantikan matriks-matriks ini dalam masalah pengoptimuman di atas, anda memperoleh Hamiltonian berikut

$$
H_C=\sum_{ij}Q_{ij}Z_iZ_j + \sum_i b_iZ_i.
$$

*Ingat juga bahawa matriks $Z$ tertanam dalam ruang komputasi komputer kuantum, iaitu ruang Hilbert bersaiz $2^n\times 2^n$. Oleh itu, anda harus memahami sebutan seperti $Z_iZ_j$ sebagai hasil darab tensor $Z_i\otimes Z_j$ yang tertanam dalam ruang Hilbert $2^n\times 2^n$. Sebagai contoh, dalam masalah dengan lima pemboleh ubah keputusan, sebutan $Z_1Z_3$ difahami bermaksud $I\otimes Z_3\otimes I\otimes Z_1\otimes I$ di mana $I$ ialah matriks identiti $2\times 2$.*
</details>

Hamiltonian ini dipanggil **Hamiltonian fungsi kos**. Ia mempunyai sifat bahawa keadaan asasnya sepadan dengan penyelesaian yang **meminimumkan fungsi kos $f(x)$**.
Oleh itu, untuk menyelesaikan masalah pengoptimuman anda, kini anda perlu menyediakan keadaan asas $H_C$ (atau keadaan dengan pertindihan tinggi dengannya) pada komputer kuantum. Kemudian, mengambil sampel daripada keadaan ini akan, dengan kebarangkalian tinggi, menghasilkan penyelesaian kepada $min~f(x)$.
Sekarang mari kita pertimbangkan Hamiltonian $H_C$ untuk masalah **Max-Cut**. Biarkan setiap bucu graf dikaitkan dengan qubit dalam keadaan $|0\rangle$ atau $|1\rangle$, di mana nilai menandakan set yang dimasuki bucu tersebut. Matlamat masalah adalah untuk memaksimumkan bilangan tepi $(v_1, v_2)$ di mana $v_1 = |0\rangle$ dan $v_2 = |1\rangle$, atau sebaliknya. Jika kita kaitkan operator $Z$ dengan setiap qubit, di mana

$$
    Z|0\rangle = |0\rangle \qquad Z|1\rangle = -|1\rangle
$$

maka tepi $(v_1, v_2)$ termasuk dalam potongan jika nilai eigen $(Z_1|v_1\rangle) \cdot (Z_2|v_2\rangle) = -1$; dengan kata lain, qubit yang dikaitkan dengan $v_1$ dan $v_2$ adalah berbeza. Begitu juga, $(v_1, v_2)$ tidak termasuk dalam potongan jika nilai eigen $(Z_1|v_1\rangle) \cdot (Z_2|v_2\rangle) = 1$. Perhatikan bahawa kita tidak mempedulikan keadaan qubit tepat yang dikaitkan dengan setiap bucu, sebaliknya kita hanya mempedulikan sama ada ia sama atau tidak merentasi satu tepi. Masalah Max-Cut memerlukan kita mencari penetapan qubit pada bucu-bucu supaya nilai eigen Hamiltonian berikut diminimumkan
$$
    H_C = \sum_{(i,j) \in e} Q_{ij} \cdot Z_i Z_j.
$$

Dengan kata lain, $b_i = 0$ untuk semua $i$ dalam masalah Max-Cut. Nilai $Q_{ij}$ menandakan berat tepi. Dalam tutorial ini kita mempertimbangkan graf tidak berwajaran, iaitu $Q_{ij} = 1.0$ untuk semua $i, j$.

In [ ]:
def build_max_cut_paulis(
    graph: rx.PyGraph,
) -> list[tuple[str, list[int], float]]:
    """Convert the graph to Pauli list.

    This function does the inverse of `build_max_cut_graph`
    """
    pauli_list = []
    for edge in list(graph.edge_list()):
        weight = graph.get_edge_data(edge[0], edge[1])
        pauli_list.append(("ZZ", [edge[0], edge[1]], weight))
    return pauli_list


max_cut_paulis = build_max_cut_paulis(graph)
cost_hamiltonian = SparsePauliOp.from_sparse_list(max_cut_paulis, n)
print("Cost Function Hamiltonian:", cost_hamiltonian)

Cost Function Hamiltonian: SparsePauliOp(['IIIZZ', 'IIZIZ', 'ZIIIZ', 'IIZZI', 'IZZII', 'ZZIII'],
              coeffs=[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j])


#### Hamiltonian &rarr; quantum circuit

The Hamiltonian $H_C$ contains the quantum definition of your problem. Now you can create a quantum circuit that will help *sample* good solutions from the quantum computer. The QAOA is inspired by quantum annealing and applies alternating layers of operators in the quantum circuit.

The general idea is to start in the ground state of a known system, $H^{\otimes n}|0\rangle$ above, and then steer the system into the ground state of the cost operator that you are interested in. This is done by applying the operators $\exp\{-i\gamma_k H_C\}$ and $\exp\{-i\beta_k H_m\}$ with angles $\gamma_1,...,\gamma_p$ and $\beta_1,...,\beta_p~$.


The quantum circuit that you generate is **parametrized** by $\gamma_i$ and $\beta_i$, so you can try out different values of $\gamma_i$ and $\beta_i$ and sample from the resulting state.

![Circuit diagram with QAOA layers](../docs/images/tutorials/quantum-approximate-optimization-algorithm/circuit-diagram.svg)


In this case, you will try an example with one QAOA layer that contains two parameters: $\gamma_1$ and $\beta_1$.

In [4]:
circuit = QAOAAnsatz(cost_operator=cost_hamiltonian, reps=2)
circuit.measure_all()

circuit.draw("mpl")

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/7bd8c6d4-f40f-4a11-a440-0b26d9021b53-0.avif" alt="Output of the previous code cell" />

In [5]:
circuit.parameters

ParameterView([ParameterVectorElement(β[0]), ParameterVectorElement(β[1]), ParameterVectorElement(γ[0]), ParameterVectorElement(γ[1])])

### Step 2: Optimize problem for quantum hardware execution

The circuit above contains a series of abstractions useful to think about quantum algorithms, but not possible to run on the hardware. To be able to run on a QPU, the circuit needs to undergo a series of operations that make up the **transpilation** or **circuit optimization** step of the pattern.

The Qiskit library offers a series of **transpilation passes** that cater to a wide range of circuit transformations. You need to make sure that your circuit is **optimized** for your purpose.

Transpilation may involves several steps, such as:

* **Initial mapping** of the qubits in the circuit (such as decision variables) to physical qubits on the device.
* **Unrolling** of the instructions in the quantum circuit to the hardware-native instructions that the backend understands.
* **Routing** of any qubits in the circuit that interact to physical qubits that are adjacent with one another.
* **Error suppression** by adding single-qubit gates to suppress noise with dynamical decoupling.


More information about transpilation is available in our [documentation](/docs/guides/transpile).

The following code transforms and optimizes the abstract circuit into a format that is ready for execution on one of devices accessible through the cloud using the **Qiskit IBM Runtime service**.

In [6]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
print(backend)

# Create pass manager for transpilation
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)

candidate_circuit = pm.run(circuit)
candidate_circuit.draw("mpl", fold=False, idle_wires=False)

<IBMBackend('test_heron_pok_1')>


<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/3f28a422-805c-4d3d-b5f6-62539e9133bd-1.avif" alt="Output of the previous code cell" />

### Step 3: Execute using Qiskit primitives

In the QAOA workflow, the optimal QAOA parameters are found in an iterative optimization loop, which runs a series of circuit evaluations and uses a classical optimizer to find the optimal $\beta_k$ and $\gamma_k$ parameters. This execution loop is executed via the following steps:

1. Define the initial parameters
2. Instantiate a new `Session` containing the optimization loop and the primitive used to sample the circuit
3. Once an optimal set of parameters is found, execute the circuit a final time to obtain a final distribution which will be used in the post-process step.

#### Define circuit with initial parameters
We start with arbitrary chosen parameters.

In [7]:
initial_gamma = np.pi
initial_beta = np.pi / 2
init_params = [initial_beta, initial_beta, initial_gamma, initial_gamma]

### Langkah 2: Optimumkan masalah untuk pelaksanaan perkakasan kuantum
Litar di atas mengandungi siri abstraksi yang berguna untuk berfikir tentang algoritma kuantum, tetapi tidak boleh dijalankan pada perkakasan. Untuk dapat dijalankan pada QPU, litar perlu melalui siri operasi yang membentuk langkah **transpilasi** atau **pengoptimuman litar** dalam corak tersebut.

Perpustakaan Qiskit menawarkan siri **laluan transpilasi** yang melayani pelbagai transformasi litar. Anda perlu memastikan litar anda **dioptimumkan** untuk tujuan anda.

Transpilasi mungkin melibatkan beberapa langkah, seperti:

* **Pemetaan awal** qubit dalam litar (seperti pemboleh ubah keputusan) kepada qubit fizikal pada peranti.
* **Pengembangan** arahan dalam litar kuantum kepada arahan natif perkakasan yang difahami oleh Backend.
* **Penghalauan** mana-mana qubit dalam litar yang berinteraksi kepada qubit fizikal yang bersebelahan antara satu sama lain.
* **Penindasan ralat** dengan menambah get satu-qubit untuk menindas hingar dengan penyahgandingan dinamik.

Maklumat lanjut tentang transpilasi tersedia dalam [dokumentasi](/guides/transpile) kami.

Kod berikut mengubah dan mengoptimumkan litar abstrak kepada format yang sedia untuk dilaksanakan pada salah satu peranti yang boleh diakses melalui awan menggunakan **perkhidmatan Qiskit IBM Runtime**.

In [8]:
def cost_func_estimator(params, ansatz, hamiltonian, estimator):
    # transform the observable defined on virtual qubits to
    # an observable defined on all physical qubits
    isa_hamiltonian = hamiltonian.apply_layout(ansatz.layout)

    pub = (ansatz, isa_hamiltonian, params)
    job = estimator.run([pub])

    results = job.result()[0]
    cost = results.data.evs

    objective_func_vals.append(cost)

    return cost

In [9]:
objective_func_vals = []  # Global variable
with Session(backend=backend) as session:
    # If using qiskit-ibm-runtime<0.24.0, change `mode=` to `session=`
    estimator = Estimator(mode=session)
    estimator.options.default_shots = 1000

    # Set simple error suppression/mitigation options
    estimator.options.dynamical_decoupling.enable = True
    estimator.options.dynamical_decoupling.sequence_type = "XY4"
    estimator.options.twirling.enable_gates = True
    estimator.options.twirling.num_randomizations = "auto"

    result = minimize(
        cost_func_estimator,
        init_params,
        args=(candidate_circuit, cost_hamiltonian, estimator),
        method="COBYLA",
        tol=1e-2,
    )
    print(result)

 message: Return from COBYLA because the trust region radius reaches its lower bound.
 success: True
  status: 0
     fun: -1.6295230263157894
       x: [ 1.530e+00  1.439e+00  4.071e+00  4.434e+00]
    nfev: 26
   maxcv: 0.0


![Output of the previous code cell](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/3f28a422-805c-4d3d-b5f6-62539e9133bd-1.avif)

### Langkah 3: Laksanakan menggunakan primitif Qiskit
Dalam aliran kerja QAOA, parameter QAOA optimum ditemui dalam gelung pengoptimuman iteratif, yang menjalankan siri penilaian litar dan menggunakan pengoptimum klasik untuk mencari parameter $\beta_k$ dan $\gamma_k$ yang optimum. Gelung pelaksanaan ini dilaksanakan melalui langkah-langkah berikut:

1. Takrifkan parameter awal
2. Mulakan Session baharu yang mengandungi gelung pengoptimuman dan primitif yang digunakan untuk mengambil sampel litar
3. Setelah set parameter optimum ditemui, laksanakan litar sekali lagi untuk mendapatkan taburan akhir yang akan digunakan dalam langkah pasca-proses.
#### Takrifkan litar dengan parameter awal
Kita mulakan dengan parameter yang dipilih secara sewenang-wenangnya.

In [10]:
plt.figure(figsize=(12, 6))
plt.plot(objective_func_vals)
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.show()

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/e14ecc92-0.avif" alt="Output of the previous code cell" />

#### Takrifkan Backend dan primitif pelaksanaan
Gunakan **primitif Qiskit Runtime** untuk berinteraksi dengan Backend IBM&reg;. Dua primitif ialah Sampler dan Estimator, dan pilihan primitif bergantung pada jenis pengukuran yang ingin anda jalankan pada komputer kuantum. Untuk meminimumkan $H_C$, gunakan Estimator kerana pengukuran fungsi kos hanyalah nilai jangkaan $\langle H_C \rangle$.
#### Jalankan
Primitif menawarkan pelbagai [mod pelaksanaan](/guides/execution-modes) untuk menjadualkan beban kerja pada peranti kuantum, dan aliran kerja QAOA berjalan secara iteratif dalam sesi.

![Ilustrasi menunjukkan tingkah laku mod jalan Kerja Tunggal, Kelompok, dan Sesi.](../docs/images/tutorials/quantum-approximate-optimization-algorithm/runtime-modes.avif)

Anda boleh memasukkan fungsi kos berasaskan Sampler ke dalam rutin peminimuman SciPy untuk mencari parameter optimum.

In [11]:
optimized_circuit = candidate_circuit.assign_parameters(result.x)
optimized_circuit.draw("mpl", fold=False, idle_wires=False)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/2989e76e-4296-4dd8-b065-2b8fced064cf-0.avif" alt="Output of the previous code cell" />

In [12]:
# If using qiskit-ibm-runtime<0.24.0, change `mode=` to `backend=`
sampler = Sampler(mode=backend)
sampler.options.default_shots = 10000

# Set simple error suppression/mitigation options
sampler.options.dynamical_decoupling.enable = True
sampler.options.dynamical_decoupling.sequence_type = "XY4"
sampler.options.twirling.enable_gates = True
sampler.options.twirling.num_randomizations = "auto"

pub = (optimized_circuit,)
job = sampler.run([pub], shots=int(1e4))
counts_int = job.result()[0].data.meas.get_int_counts()
counts_bin = job.result()[0].data.meas.get_counts()
shots = sum(counts_int.values())
final_distribution_int = {key: val / shots for key, val in counts_int.items()}
final_distribution_bin = {key: val / shots for key, val in counts_bin.items()}
print(final_distribution_int)

{28: 0.0328, 11: 0.0343, 2: 0.0296, 25: 0.0308, 16: 0.0303, 27: 0.0302, 13: 0.0323, 7: 0.0312, 4: 0.0296, 9: 0.0295, 26: 0.0321, 30: 0.031, 23: 0.0324, 31: 0.0303, 21: 0.0335, 15: 0.0317, 12: 0.0309, 29: 0.0297, 3: 0.0313, 5: 0.0312, 6: 0.0274, 10: 0.0329, 22: 0.0353, 0: 0.0315, 20: 0.0326, 8: 0.0322, 14: 0.0306, 17: 0.0295, 18: 0.0279, 1: 0.0325, 24: 0.0334, 19: 0.0295}


### Step 4: Post-process and return result in desired classical format

The post-processing step interprets the sampling output to return a solution for your original problem. In this case, you are interested in the bitstring with the highest probability as this determines the optimal cut. The symmetries in the problem allow for four possible solutions, and the sampling process will return one of them with a slightly higher probability, but you can see in the plotted distribution below that four of the bitstrings are distinctively more likely than the rest.

In [13]:
# auxiliary functions to sample most likely bitstring
def to_bitstring(integer, num_bits):
    result = np.binary_repr(integer, width=num_bits)
    return [int(digit) for digit in result]


keys = list(final_distribution_int.keys())
values = list(final_distribution_int.values())
most_likely = keys[np.argmax(np.abs(values))]
most_likely_bitstring = to_bitstring(most_likely, len(graph))
most_likely_bitstring.reverse()

print("Result bitstring:", most_likely_bitstring)

Result bitstring: [0, 1, 1, 0, 1]


In [14]:
matplotlib.rcParams.update({"font.size": 10})
final_bits = final_distribution_bin
values = np.abs(list(final_bits.values()))
top_4_values = sorted(values, reverse=True)[:4]
positions = []
for value in top_4_values:
    positions.append(np.where(values == value)[0])
fig = plt.figure(figsize=(11, 6))
ax = fig.add_subplot(1, 1, 1)
plt.xticks(rotation=45)
plt.title("Result Distribution")
plt.xlabel("Bitstrings (reversed)")
plt.ylabel("Probability")
ax.bar(list(final_bits.keys()), list(final_bits.values()), color="tab:grey")
for p in positions:
    ax.get_children()[int(p[0])].set_color("tab:purple")
plt.show()

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/650875e9-adbc-43bd-9505-556be2566278-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/e14ecc92-0.avif)

Setelah anda menemui parameter optimum untuk litar, anda boleh menetapkan parameter ini dan mengambil sampel taburan akhir yang diperoleh dengan parameter yang dioptimumkan. Di sinilah primitif *Sampler* harus digunakan kerana ia adalah taburan kebarangkalian rentetan bit yang sepadan dengan potongan optimum graf.

**Nota:** Ini bermakna menyediakan keadaan kuantum $\psi$ dalam komputer dan kemudian mengukurnya. Pengukuran akan meruntuhkan keadaan kepada satu keadaan asas komputasi tunggal - contohnya, `010101110000...` - yang sepadan dengan penyelesaian calon $x$ kepada masalah pengoptimuman asal kita ($\max f(x)$ atau $\min f(x)$ bergantung kepada tugas).

In [15]:
# auxiliary function to plot graphs
def plot_result(G, x):
    colors = ["tab:grey" if i == 0 else "tab:purple" for i in x]
    pos, _default_axes = rx.spring_layout(G), plt.axes(frameon=True)
    rx.visualization.mpl_draw(
        G, node_color=colors, node_size=100, alpha=0.8, pos=pos
    )


plot_result(graph, most_likely_bitstring)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/33135970-8bc4-4fb2-ab87-08726a432ce4-0.avif" alt="Output of the previous code cell" />

And calculate the value of the cut:

In [16]:
def evaluate_sample(x: Sequence[int], graph: rx.PyGraph) -> float:
    assert len(x) == len(
        list(graph.nodes())
    ), "The length of x must coincide with the number of nodes in the graph."
    return sum(
        x[u] * (1 - x[v]) + x[v] * (1 - x[u])
        for u, v in list(graph.edge_list())
    )


cut_value = evaluate_sample(most_likely_bitstring, graph)
print("The value of the cut is:", cut_value)

The value of the cut is: 5


## Part II. Scale it up!

You have access to many devices with over 100 qubits on IBM Quantum&reg; Platform. Select one on which to solve Max-Cut on a 100-node weighted graph. This is a "utility-scale" problem. The steps to build the workflow are followed the same as above, but with a much larger graph.

In [17]:
n = 100  # Number of nodes in graph
graph_100 = rx.PyGraph()
graph_100.add_nodes_from(np.arange(0, n, 1))
elist = []
for edge in backend.coupling_map:
    if edge[0] < n and edge[1] < n:
        elist.append((edge[0], edge[1], 1.0))
graph_100.add_edges_from(elist)
draw_graph(graph_100, node_size=200, with_labels=True, width=1)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/590fe2ce-0.avif" alt="Output of the previous code cell" />

### Langkah 4: Pasca-proses dan kembalikan keputusan dalam format klasik yang dikehendaki
Langkah pasca-proses mentafsirkan output pensampelan untuk mengembalikan penyelesaian bagi masalah asal anda. Dalam kes ini, anda berminat dengan rentetan bit yang mempunyai kebarangkalian tertinggi kerana ini menentukan potongan optimum. Simetri dalam masalah membolehkan empat penyelesaian yang mungkin, dan proses pensampelan akan mengembalikan salah satunya dengan kebarangkalian yang sedikit lebih tinggi, tetapi anda dapat lihat dalam taburan yang diplot di bawah bahawa empat rentetan bit jelas lebih mungkin berbanding yang lain.

In [18]:
max_cut_paulis_100 = build_max_cut_paulis(graph_100)

cost_hamiltonian_100 = SparsePauliOp.from_sparse_list(max_cut_paulis_100, 100)
print("Cost Function Hamiltonian:", cost_hamiltonian_100)

Cost Function Hamiltonian: SparsePauliOp(['...'],
              coeffs=[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0

#### Hamiltonian &rarr; quantum circuit

In [19]:
circuit_100 = QAOAAnsatz(cost_operator=cost_hamiltonian_100, reps=1)
circuit_100.measure_all()

circuit_100.draw("mpl", fold=False, scale=0.2, idle_wires=False)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/9693adfc-0.avif" alt="Output of the previous code cell" />

### Step 2: Optimize problem for quantum execution
To scale the circuit optimization step to utility-scale problems, you can take advantage of the high performance transpilation strategies introduced in Qiskit SDK v1.0. Other tools include the new transpiler service with [AI enhanced transpiler passes](/docs/guides/ai-transpiler-passes).

In [20]:
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)

candidate_circuit_100 = pm.run(circuit_100)
candidate_circuit_100.draw("mpl", fold=False, scale=0.1, idle_wires=False)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/3a14e7ad-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/650875e9-adbc-43bd-9505-556be2566278-0.avif)

#### Visualisasikan potongan terbaik
Daripada rentetan bit optimum, anda kemudian boleh menggambarkan potongan ini pada graf asal.

In [21]:
initial_gamma = np.pi
initial_beta = np.pi / 2
init_params = [initial_beta, initial_gamma]

objective_func_vals = []  # Global variable
with Session(backend=backend) as session:
    # If using qiskit-ibm-runtime<0.24.0, change `mode=` to `session=`
    estimator = Estimator(mode=session)

    estimator.options.default_shots = 1000

    # Set simple error suppression/mitigation options
    estimator.options.dynamical_decoupling.enable = True
    estimator.options.dynamical_decoupling.sequence_type = "XY4"
    estimator.options.twirling.enable_gates = True
    estimator.options.twirling.num_randomizations = "auto"

    result = minimize(
        cost_func_estimator,
        init_params,
        args=(candidate_circuit_100, cost_hamiltonian_100, estimator),
        method="COBYLA",
    )
    print(result)

 message: Return from COBYLA because the trust region radius reaches its lower bound.
 success: True
  status: 0
     fun: -3.9939191365979383
       x: [ 1.571e+00  3.142e+00]
    nfev: 29
   maxcv: 0.0


![Output of the previous code cell](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/33135970-8bc4-4fb2-ab87-08726a432ce4-0.avif)

Dan kira nilai potongan:

In [22]:
optimized_circuit_100 = candidate_circuit_100.assign_parameters(result.x)
optimized_circuit_100.draw("mpl", fold=False, idle_wires=False)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/1c432c2e-0.avif" alt="Output of the previous code cell" />

Finally, execute the circuit with the optimal parameters to sample from the corresponding distribution.

In [23]:
# If using qiskit-ibm-runtime<0.24.0, change `mode=` to `backend=`
sampler = Sampler(mode=backend)
sampler.options.default_shots = 10000

# Set simple error suppression/mitigation options
sampler.options.dynamical_decoupling.enable = True
sampler.options.dynamical_decoupling.sequence_type = "XY4"
sampler.options.twirling.enable_gates = True
sampler.options.twirling.num_randomizations = "auto"


pub = (optimized_circuit_100,)
job = sampler.run([pub], shots=int(1e4))

counts_int = job.result()[0].data.meas.get_int_counts()
counts_bin = job.result()[0].data.meas.get_counts()
shots = sum(counts_int.values())
final_distribution_100_int = {
    key: val / shots for key, val in counts_int.items()
}

## Bahagian II. Skala ia lebih besar!
Anda mempunyai akses kepada banyak peranti dengan lebih 100 qubit pada IBM Quantum&reg; Platform. Pilih salah satu untuk menyelesaikan Max-Cut pada graf berwajaran 100 nod. Ini ialah masalah "skala utiliti". Langkah-langkah untuk membina aliran kerja diikuti sama seperti di atas, tetapi dengan graf yang jauh lebih besar.

In [24]:
plt.figure(figsize=(12, 6))
plt.plot(objective_func_vals)
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.show()

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/0fda3611-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/590fe2ce-0.avif)

### Langkah 1: Petakan input klasik kepada masalah kuantum
#### Graf &rarr; Hamiltonian
Pertama, tukar graf yang ingin anda selesaikan terus kepada Hamiltonian yang sesuai untuk QAOA.

In [25]:
_PARITY = np.array(
    [-1 if bin(i).count("1") % 2 else 1 for i in range(256)],
    dtype=np.complex128,
)


def evaluate_sparse_pauli(state: int, observable: SparsePauliOp) -> complex:
    """Utility for the evaluation of the expectation value of a measured state."""
    packed_uint8 = np.packbits(observable.paulis.z, axis=1, bitorder="little")
    state_bytes = np.frombuffer(
        state.to_bytes(packed_uint8.shape[1], "little"), dtype=np.uint8
    )
    reduced = np.bitwise_xor.reduce(packed_uint8 & state_bytes, axis=1)
    return np.sum(observable.coeffs * _PARITY[reduced])


def best_solution(samples, hamiltonian):
    """Find solution with lowest cost"""
    min_cost = 1000
    min_sol = None
    for bit_str in samples.keys():
        # Qiskit use little endian hence the [::-1]
        candidate_sol = int(bit_str)
        # fval = qp.objective.evaluate(candidate_sol)
        fval = evaluate_sparse_pauli(candidate_sol, hamiltonian).real
        if fval <= min_cost:
            min_sol = candidate_sol

    return min_sol


best_sol_100 = best_solution(final_distribution_100_int, cost_hamiltonian_100)
best_sol_bitstring_100 = to_bitstring(int(best_sol_100), len(graph_100))
best_sol_bitstring_100.reverse()

print("Result bitstring:", best_sol_bitstring_100)

Result bitstring: [1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1]


Next, visualize the cut. Nodes of the same color belong to the same group.

In [26]:
plot_result(graph_100, best_sol_bitstring_100)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/b4a25e28-0.avif" alt="Output of the previous code cell" />

#### Hamiltonian &rarr; litar kuantum

In [27]:
cut_value_100 = evaluate_sample(best_sol_bitstring_100, graph_100)
print("The value of the cut is:", cut_value_100)

The value of the cut is: 124


![Output of the previous code cell](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/9693adfc-0.avif)

### Langkah 2: Optimumkan masalah untuk pelaksanaan kuantum
Untuk menskalakan langkah pengoptimuman litar kepada masalah skala utiliti, anda boleh memanfaatkan strategi transpilasi berprestasi tinggi yang diperkenalkan dalam Qiskit SDK v1.0. Alat lain termasuk perkhidmatan Transpiler baharu dengan [laluan Transpiler dipertingkat AI](/guides/ai-transpiler-passes).

In [28]:
# auxiliary function to help plot cumulative distribution functions
def _plot_cdf(objective_values: dict, ax, color):
    x_vals = sorted(objective_values.keys(), reverse=True)
    y_vals = np.cumsum([objective_values[x] for x in x_vals])
    ax.plot(x_vals, y_vals, color=color)


def plot_cdf(dist, ax, title):
    _plot_cdf(
        dist,
        ax,
        "C1",
    )
    ax.vlines(min(list(dist.keys())), 0, 1, "C1", linestyle="--")

    ax.set_title(title)
    ax.set_xlabel("Objective function value")
    ax.set_ylabel("Cumulative distribution function")
    ax.grid(alpha=0.3)


# auxiliary function to convert bit-strings to objective values
def samples_to_objective_values(samples, hamiltonian):
    """Convert the samples to values of the objective function."""

    objective_values = defaultdict(float)
    for bit_str, prob in samples.items():
        candidate_sol = int(bit_str)
        fval = evaluate_sparse_pauli(candidate_sol, hamiltonian).real
        objective_values[fval] += prob

    return objective_values

In [29]:
result_dist = samples_to_objective_values(
    final_distribution_100_int, cost_hamiltonian_100
)

Finally, you can plot the cumulative distribution function to visualize how each sample contributes to the total probability distribution and the corresponding objective value. The horizontal spread shows the range of objective values of the samples in the final distribution. Ideally, you would see that the cumulative distribution function has "jumps" at the lower end of the objective function value axis. This would mean that few solutions with low cost have high probability of being sampled. A smooth, wide curve indicates that each sample is similarly likely, and they can have very different objective values, low or high.

In [30]:
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
plot_cdf(result_dist, ax, "Eagle device")

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/4381a2b3-0.avif" alt="Output of the previous code cell" />

Setelah parameter optimum daripada menjalankan QAOA pada peranti ditemui, tetapkan parameter pada litar.